# Smaller Mask Extraction

> smaller mask pin extraction

In [ ]:
#| default_exp extract_smaller_mask

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import cv2
from typing import NewType, List, Tuple, Union, Dict
from pathlib import Path
from scipy.ndimage import label
from tqdm.auto import tqdm
from argparse import ArgumentParser, BooleanOptionalAction

In [ ]:
#| export
OpenCvImage = NewType('OpenCvImage', np.ndarray)

In [ ]:
#| export
CV_TOOLS = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/cv_tools')
sys.path.append(str(CV_TOOLS))

# %% ../../../nbs/39_preprocessing.zero_degree_solder_pin.ipynb 6
custom_lib_path = Path(r'/home/ai_warstein/homes/goni/custom_libs')
CURRENT_REPO = Path(r'/home/ai_sintercra/homes/hasan/projects/git_data/new_test')
sys.path.append(str(custom_lib_path))
sys.path.append(str(CURRENT_REPO))
from dotenv import load_dotenv


In [ ]:
#| export
from cv_tools.core import *

In [ ]:
#| export
from new_test.core import *
from new_test.pin_map import *

In [ ]:
#| export    
def get_full_image_smaller_mask(
        img:OpenCvImage,# actual image
        msk:OpenCvImage, # mask where pins are small or no pins are found
        im_name:str,
        recipe_no:str,
        col_bbox:List[Tuple[int, int, int, int]],
        corrected_msk:OpenCvImage=None, # In case if from other model or any how if corrected masks is also available
        save_im_path:Union[str, None]=None,
        save_msk_path:Union[str, None]=None,
        save_corrected_msk_path:Union[str, None]=None,
        minimum_area:int=9000
   )->Tuple[OpenCvImage, OpenCvImage]:
  """
  Get image of all the columns and saving pins where the mask is smaller than specific number
  """
  # getting bounding boxes for each column

  if save_im_path is not None:
    Path(save_im_path).mkdir(exist_ok=True, parents=True)

  if save_msk_path is not None:
    Path(save_msk_path).mkdir(exist_ok=True, parents=True)

  if save_corrected_msk_path is not None:
    Path(save_corrected_msk_path).mkdir(exist_ok=True, parents=True)

  num_cols_ = split_image_no()[recipe_no]
  pin_map_func =f'pin_map{recipe_no}'

  last_mask = np.zeros_like(img)
  pin_index_func=globals()[pin_map_func]
  all_cols = []
  num_cols = len(pin_index_func().keys())

  for col_no in range(num_cols):
        col_bbox_coord = col_bbox[col_no]

        col_data = img[col_bbox_coord[0]:col_bbox_coord[1], col_bbox_coord[2]:col_bbox_coord[3]]
        col_data_mask = msk[col_bbox_coord[0]:col_bbox_coord[1], col_bbox_coord[2]:col_bbox_coord[3]]
        if corrected_msk is not None:
          col_data_corrected_mask = corrected_msk[col_bbox_coord[0]:col_bbox_coord[1], col_bbox_coord[2]:col_bbox_coord[3]]

        pin_indexes = pin_index_func()[col_no]
        # getting image and mask part
        single_col_parts = split_image(
                            col_data, 
                            num_cols_, 
                            'vertical')

        single_col_parts_msk = split_image(
                      col_data_mask, 
                      num_cols_, 
                      'vertical')
        if corrected_msk is not None:
          single_col_parts_corrected_msk = split_image(col_data_corrected_mask, num_cols_, 'vertical')

        parts=[]
        for idx, part in enumerate(single_col_parts):
            if idx in pin_indexes:
                part = part
                msk_part = single_col_parts_msk[idx]
                _, n_f = label(msk_part)
                if (save_im_path is not None and cv2.countNonZero(msk_part)<minimum_area)\
                  or (save_im_path is not None and n_f>1):
                    cv2.imwrite(
                       f'{save_im_path}/{Path(im_name).stem}_col_{col_no}_{idx}.png', 
                       part)
                    if save_msk_path is not None:
                        cv2.imwrite(
                           f'{save_msk_path}/{Path(im_name).stem}_col_{col_no}_{idx}.png', 
                           msk_part)
                    if save_corrected_msk_path is not None:
                        cv2.imwrite(
                           f'{save_corrected_msk_path}/{Path(im_name).stem}_col_{col_no}_{idx}.png',
                             single_col_parts_corrected_msk[idx])
            else:
                #TODO: add pin mask
                part = np.zeros_like(part)
            parts.append(part)
        sn_col=np.concatenate(parts, axis=0)

        last_mask[col_bbox_coord[0]:col_bbox_coord[1], col_bbox_coord[2]:col_bbox_coord[3]] = sn_col
        all_cols.append(sn_col)
  full_part = np.concatenate(all_cols, axis=1)
  return last_mask, full_part


In [ ]:
#| export
def extract_smaller_mask_pin(
        im_path:str,
        msk_path:str,
        corrected_msk_path:str,
        tmp_path:str,
        recipe_no:str,
        save_im_path:Union[str, Path]=None,
        save_msk_path:Union[str, Path]=None,
        save_corrected_msk_path:Union[str, Path]=None,
        min_msk_area:int=300,      
    ):
    'Check mask of each pin, if it is smaller than specific area, save it to the specific folder'
    
    # getting all  images from path
    img = read_img(im_path)
    msk = read_img(msk_path)
    cor_msk = read_img(corrected_msk_path)

    tmp_img = read_img(tmp_path)
    
    # getting bouding box based on recipe 
    bbox_func = globals()[f'get_all_columns_{recipe_no}']
    col_bbox = bbox_func(
        tst_img=img,
        tmp_img=tmp_img
    )
    # Now saving all the parts which has smaller mask


    zoomed_, full_ = get_full_image_smaller_mask(
        img=img,
        msk=msk,
        corrected_msk=cor_msk,
        im_name=Path(im_path).name,
        recipe_no=recipe_no,
        col_bbox=col_bbox,
        save_im_path=save_im_path,
        save_msk_path=save_msk_path,
        save_corrected_msk_path=save_corrected_msk_path,
        minimum_area=min_msk_area
                                )

In [ ]:
#| eval: false
MODEL='time_11_04_21_val_frGrnd0.9498_epoch_200.h5'
core = Path(fr'/home/ai_easypid.work/CurrentTrainingData20240812_trn_val/Incoming_1B_Loetstift/Incoming_1B_Loetstift_unzip/main_im2_cropped_masks/{MODEL}/failed/missing/images')
add_pin_core = Path(core.parent.parent,'additional' )
add_pin_im_path = Path(add_pin_core, 'images')
add_pin_msk_path = Path(add_pin_core, 'masks')
add_pin_overlay_path = Path(add_pin_core, 'overlay')
msk_path = Path(core.parent, 'masks')
recipe_no = '17'
tmp_path = Path(fr'/home/ai_easypid.work/CurrentTrainingData20240812_trn_val/templates/{recipe_no}.png')

In [ ]:
add_pin_path.exists(), add_pin_msk_path.exists(), add_pin_overlay_path.exists()

In [ ]:
#| eval: false
query_fn = core.ls()[0]
msk_fn = Path(msk_path, query_fn.name)
query_img = read_img(query_fn)
msk_img = read_img(msk_fn)
tmp_img = read_img(tmp_path)
pin_map_func = f'pin_map{recipe_no}'
bbox_func = globals()[f'get_all_columns_{recipe_no}']
pin_index_func=globals()[pin_map_func]
col_bbox = bbox_func(
    tst_img=query_img, 
    tmp_img = tmp_img
)
# which positon we are expecting pins
pin_positions_idx = pin_index_func()[10]
num_cols = len(pin_index_func().keys())

In [ ]:
col_bbox_coord = col_bbox[10]

col_data = query_img[col_bbox_coord[0]+15:col_bbox_coord[1], col_bbox_coord[2]-10:col_bbox_coord[3]]
show_(split_image(col_data, num_cols, direction='vertical')[0])
show_(split_image(col_data, num_cols, direction='vertical')[-1])

In [ ]:
Path(query_fn).name

In [ ]:
zoomed_, full_ = get_full_image_smaller_mask(
        img=query_img,
        msk=msk_img,
        corrected_msk=None,
        im_name=Path(query_fn).name,
        recipe_no=recipe_no,
        col_bbox=col_bbox,
            save_im_path=None,
        save_msk_path=None,
        save_corrected_msk_path=None,
        minimum_area=30
                                )

In [ ]:
show_(zoomed_)

# Additional pin query image

In [ ]:
#| hide
#recipe_name='17'
#im_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/main_im2_cropped_masks_new/time_13_02_08_val_frGrnd0.9557_epoch_180_missing_pins/images_masks_new/time_11_18_24_val_frGrnd0.9470_epoch_185_additional_pins/images')
#msk_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/main_im2_cropped_masks_new/time_13_02_08_val_frGrnd0.9557_epoch_180_missing_pins/images_masks_new/time_11_18_24_val_frGrnd0.9470_epoch_185_additional_pins/images_masks_new/time_13_02_08_val_frGrnd0.9557_epoch_180_missing_pins/masks')
#cor_msk_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/main_im2_cropped_masks_new/time_13_02_08_val_frGrnd0.9557_epoch_180_missing_pins/images_masks_new/time_11_18_24_val_frGrnd0.9470_epoch_185_additional_pins/corrected_masks')
#save_im_path=Path(im_path.parent, 'sn_img')
#save_msk_path=Path(im_path.parent, 'sn_msk')
#save_cor_msk_path = Path(im_path.parent, 'sn_cor_msk')
#TMP_PATH = os.getenv('TEMP_PATH')
#tmp_path = Path(TMP_PATH, f'{recipe_name}.png')
#im_fn = im_path.ls()[0]
#fn_ = Path(im_fn).name
#msk_fn = Path(msk_path, fn_)
#msk_fn_c = Path(cor_msk_path, fn_)

In [ ]:
#| hide
extract_smaller_mask_pin(
    im_path=im_fn,
    msk_path=msk_fn,
    corrected_msk_path=msk_fn_c,
    tmp_path=tmp_path,
    recipe_no=recipe_name,
    save_im_path=save_im_path,
    save_msk_path=save_msk_path,
    save_corrected_msk_path=save_cor_msk_path,
    min_msk_area=330
)

In [ ]:
#| export
def extract_smaller_mask_pin_folder(
    im_path:Union[str, Path],
    msk_path:Union[str, Path],
    cor_msk_path:Union[str, Path], # corrected mask path
    save_im_path:Union[str, Path]=None,
    save_msk_path:Union[str, Path]=None,
    save_cor_msk_path:Union[str, Path]=None,
    min_msk_area:int=330
   ):
    """
    Extract path from folder
    """
    images = Path(im_path).ls()
    im_names = get_name_(images)




    print(f' NUmber of images found = {len(im_names)}')
    for i in tqdm(im_names, total=len(im_names)):
        print(i)
        im_ = Path(im_path, i)
        recipe_name = get_m_name(i)
        TMP_PATH = os.getenv('TEMP_PATH')
        tmp_path=Path(TMP_PATH, f'{recipe_name}.png')
        msk_ = Path(msk_path, i)
        cor_msk_path = Path(cor_msk_path, i)
        extract_smaller_mask_pin(
            im_path=im_,
            msk_path=msk_,
            corrected_msk_path=cor_msk_path,
            tmp_path=tmp_path,
            recipe_no=recipe_name,
            min_msk_area=min_msk_area,
            save_msk_path=save_msk_path,
            save_corrected_msk_path=save_cor_msk_path,
            save_im_path=save_im_path
        )

In [ ]:
#| hide
#extract_smaller_mask_pin_folder(
    #im_path=im_path,
    #msk_path=msk_path,
    #cor_msk_path=cor_msk_path,
    #min_msk_area=300,
    #save_im_path=save_im_path,
    #save_msk_path=save_msk_path,
    #save_cor_msk_path=save_cor_msk_path
#)

In [ ]:
#| export    
def small_mask_where_no_pin_expected(
        img:OpenCvImage,# actual image
        msk:OpenCvImage, # mask where pins are small or no pins are found
        im_name:str,
        recipe_no:str,
        col_bbox:list[tuple[int, int, int, int]],
        save_im_path:Union[str, None]=None,
        minimum_area:int=1,  # minimum area of mask to be saved: normally no mask is expcted
        show_p:bool=False # whether to show the image part or not :debugging purpose
        )->tuple[OpenCvImage, OpenCvImage]:
  """
  Get image part where no mask is expected and save them
  """
  # getting bounding boxes for each column

  if save_im_path is not None:
    Path(save_im_path).mkdir(exist_ok=True, parents=True)

  num_cols_ = split_image_no()[recipe_no]
  pin_map_func =f'pin_map{recipe_no}'

  pin_index_func=globals()[pin_map_func]
  num_cols = len(pin_index_func().keys())

  for col_no in range(num_cols):
        col_bbox_coord = col_bbox[col_no]

        col_data = img[col_bbox_coord[0]:col_bbox_coord[1], col_bbox_coord[2]:col_bbox_coord[3]]
        col_data_mask = msk[col_bbox_coord[0]:col_bbox_coord[1], col_bbox_coord[2]:col_bbox_coord[3]]

        pin_indexes = pin_index_func()[col_no]
        # getting image and mask part
        single_col_parts = split_image(col_data, num_cols_, 'vertical')
        single_col_parts_msk = split_image(col_data_mask, num_cols_, 'vertical')

        parts=[]
        for idx, part in enumerate(single_col_parts):
            if idx in pin_indexes:
                part = part
            else:
                msk_part = single_col_parts_msk[idx]
                _, n_f = label(msk_part)
                if (cv2.countNonZero(msk_part)>minimum_area):
                    if show_p: 
                        show_(part)
                    if save_im_path is not None:
                        cv2.imwrite(f'{save_im_path}/{Path(im_name).stem}_col_{col_no}_{idx}.png', part)

In [ ]:
im_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/main_im2_cropped_masks_new_model/time_16_49_30_val_frGrnd0.9569_epoch_159_additional_pins/images/VFV4.7.9.5_2024022805132264_ID_00343048112818112132406_In_87_r_1_FRONT_Additional Lead_image2.png')
im_name = Path(im_path).name
msk_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/main_im2_cropped_masks_new_model/time_16_49_30_val_frGrnd0.9569_epoch_159_additional_pins/masks')
msk_fn = Path(msk_path, im_name)

In [ ]:
TMP_PATH=os.getenv('TEMP_PATH')

In [ ]:
recipe_no='87'
tmp_img = Path(TMP_PATH, f'{recipe_no}.png')
bbox_func = globals()[f'get_all_columns_{recipe_no}']
col_bbox = bbox_func(
    tst_img=read_img(im_path),
    tmp_img=read_img(tmp_img)
)

In [ ]:
small_mask_where_no_pin_expected(
        img=read_img(im_path),
        msk=read_img(msk_fn),
        im_name=im_name,
        recipe_no='87',
        col_bbox=col_bbox,
        minimum_area=1,  # minimum area of mask to be saved: normally no mask is expcted
        show_p=True) # whether to show the image part or not :debugging purpose

In [ ]:
#| export
def extract_additional_pin_part(
        im_path:str,
        msk_path:str,
        tmp_path:str,
        save_path:str,
        min_area:int=1 
        ):
    recipe_no = get_m_name(Path(im_path).name)
    tmp_img = read_img(tmp_path)
    img = read_img(im_path)
    msk_img = read_img(msk_path)
    bbox_func = globals()[f'get_all_columns_{recipe_no}']
    col_bbox = bbox_func(
        tst_img=img,
        tmp_img=tmp_img
    ) 

    small_mask_where_no_pin_expected(
        img=img,
        msk=msk_img,
        im_name=Path(im_path).name,
        recipe_no=recipe_no,
        col_bbox=col_bbox,
        minimum_area=min_area,
        save_im_path=save_path
    )

In [ ]:
#| export
def extract_additional_pin_part_folder(
        im_path:str,
        msk_path:str,
        tmp_path:str,
        save_path:str,
        min_area:int=1 
        ):
    ' Insider im_path will be looked for images and mask'
    for i in tqdm(Path(im_path).ls(),total=len(Path(im_path).ls())):
        name_ = Path(i).name
        recipe_no = get_m_name(name_)
        tmplate_path =  Path(tmp_path, f'{recipe_no}.png')
        try:    
            extract_additional_pin_part(
                im_path=i,
                msk_path=Path(msk_path, name_),
                tmp_path=tmplate_path,
                save_path=save_path,
                min_area=min_area
                )
        except Exception as e:
            print(f'Error in {name_} : {e}')



    

In [ ]:
add_im_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/main_im2_cropped_masks_new_model/time_16_49_30_val_frGrnd0.9569_epoch_159_additional_pins/images')
add_msk_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/main_im2_cropped_masks_new_model/time_16_49_30_val_frGrnd0.9569_epoch_159_additional_pins/masks')
add_im_save_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/main_im2_cropped_masks_new_model/time_16_49_30_val_frGrnd0.9569_epoch_159_additional_pins/additional_pins')
add_im_save_path.mkdir(exist_ok=True, parents=True)

extract_additional_pin_part_folder(
    im_path=add_im_path,
    msk_path=add_msk_path,
    tmp_path=TMP_PATH,
    save_path=add_im_save_path,
    min_area=1)

In [ ]:
#| export
def parse_args_():
    parser = ArgumentParser()
    parser.add_argument('--im_path', type=str, help='Path to image')
    parser.add_argument('--msk_path', type=str, help='Path to mask')
    parser.add_argument('--tmp_path', type=str, help='Path to template image, inside this folder there should be an image called recipe_no[e.g.87.png]')
    parser.add_argument('--save_path', type=str, help='Path to save the images')
    parser.add_argument('--min_area', type=int, default=1, help='Minimum area of mask to be saved')
    return parser.parse_args()

In [ ]:
#| export
def main():
    args = parse_args_()
    extract_additional_pin_part_folder(
        im_path=args.im_path,
        msk_path=args.msk_path,
        tmp_path=args.tmp_path,
        save_path=args.save_path,
        min_area=args.min_area
    )

In [ ]:
#| export
if __name__ == '__main__':
    main()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()